# Bayesian Uncertainty Minimization


In [1]:
import sys 
import numpy as np
import matplotlib.pyplot as plt
import time
import copy

sys.path.append("../..")
import gpder 
from gpder import UncertaintyOptimization
from gpder.bayes.minimizers import hybrid_minimizer

from utils import plot_gpr_evolution, plot_gpr_iterloss

In [2]:
import sys 
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error

sys.path.append("../..")
import gpder
from gpder.gaussian_process import GaussianProcessRegressor
from gpder.gaussian_process.kernels import GPKernel, GPKernelDerAware

try:
    import uproot
except:
    print("uproot must be installed to run this demo.")

In [3]:
f = uproot.open("./three_jets.root")
tree = f['tnt']

njets = np.array(tree['nj'])
njets_cut = np.where(njets < 4)

j1_threeM = np.stack((np.array(tree['j1_pt'])[njets_cut],
                      np.array(tree['j1_eta'])[njets_cut],
                      np.array(tree['j1_phi'])[njets_cut]), axis=1)

j2_threeM = np.stack((np.array(tree['j2_pt'])[njets_cut],
                      np.array(tree['j2_eta'])[njets_cut],
                      np.array(tree['j2_phi'])[njets_cut]), axis=1)

j3_threeM = np.stack((np.array(tree['j3_pt'])[njets_cut],
                      np.array(tree['j3_eta'])[njets_cut],
                      np.array(tree['j3_phi'])[njets_cut]), axis=1)

f.close()

In [4]:
def sigmoid(X):
    return 1. / (1. + np.exp(-X))

In [5]:
def Eff_MET50_sigmoid(J_scale, j_scale, 
                      j1_threeM=j1_threeM, 
                      j2_threeM=j2_threeM, 
                      j3_threeM=j3_threeM):
    count_met = 0
    count_pTcut = 0
    for i in range(len(j1_threeM)):
        # -- pT -- # 
        j1_pt = j1_threeM[i, 0] / J_scale
        j2_pt = j2_threeM[i, 0] / j_scale
        j3_pt = j3_threeM[i, 0] / j_scale
        if (j1_pt > 200 and j2_pt  < 200):
            count_pTcut += 1
            # -- pT_x decomp -- # 
            j1_pt_x = j1_pt * np.cos(j1_threeM[i, 2])
            j2_pt_x = j2_pt * np.cos(j2_threeM[i, 2])
            j3_pt_x = j3_pt * np.cos(j3_threeM[i, 2])
            # -- pT_y decomp -- #
            j1_pt_y = j1_pt * np.sin(j1_threeM[i, 2])
            j2_pt_y = j2_pt * np.sin(j2_threeM[i, 2])
            j3_pt_y = j3_pt * np.sin(j3_threeM[i, 2])
            # -- MET -- #
            met_x = j1_pt_x + j2_pt_x + j3_pt_x
            met_y = j1_pt_y + j2_pt_y + j3_pt_y
            met = np.sqrt(met_x * met_x + met_y * met_y)

            count_met += sigmoid(-(met-50.0))

    return count_met / count_pTcut

In [6]:
# -- training points -- # 
nu_a_train = [0.9, 1.0, 1.0, 1.0, 1.1]
nu_j_train = [1.0, 0.9, 1.0, 1.1, 1.0]
X_train = np.vstack((nu_a_train, nu_j_train)).T

y_train = np.array([Eff_MET50_sigmoid(X_train[i][0], 
                                      X_train[i][1]) for i in range(len(X_train))])
y_train_sig = np.array([Eff_MET50_sigmoid(X_train[i][0], 
                                          X_train[i][1]) for i in range(len(X_train))])

# -- testing points -- # 
res=10
X_lower = 0.7
X_upper = 1.3
lin = np.linspace(X_lower, X_upper, res)
nu_a_test, nu_j_test = np.meshgrid(lin, lin)
X_test = np.vstack((nu_a_test.flatten(), nu_j_test.flatten())).T 
y_test = np.array([Eff_MET50_sigmoid(X_test[i][0], X_test[i][1]) for i in range(len(X_test))])
y_test_sig = np.array([Eff_MET50_sigmoid(X_test[i][0], X_test[i][1]) for i in range(len(X_test))])

In [7]:
def dsigmoid(X):
    return sigmoid(X) * (1 - sigmoid(X))

In [8]:
def dEff_MET50_sigmoid(J_scale, j_scale, 
              j1_threeM=j1_threeM, j2_threeM=j2_threeM, j3_threeM=j3_threeM):
    dmet_dj1 = 0
    dmet_dj2j3 = 0
    count_pTcut = 0
    for i in range(len(j1_threeM)):
        # -- pT -- # 
        j1_pt = j1_threeM[i, 0] / J_scale
        j2_pt = j2_threeM[i, 0] / j_scale
        j3_pt = j3_threeM[i, 0] / j_scale
        if (j1_pt > 200 and j2_pt  < 200):
            count_pTcut += 1
            # -- pT_x decomp -- # 
            j1_pt_x = j1_pt * np.cos(j1_threeM[i, 2])
            j2_pt_x = j2_pt * np.cos(j2_threeM[i, 2])
            j3_pt_x = j3_pt * np.cos(j3_threeM[i, 2])
            # -- pT_y decomp -- #
            j1_pt_y = j1_pt * np.sin(j1_threeM[i, 2])
            j2_pt_y = j2_pt * np.sin(j2_threeM[i, 2])
            j3_pt_y = j3_pt * np.sin(j3_threeM[i, 2])
            # -- MET -- #
            met_x = j1_pt_x + j2_pt_x + j3_pt_x
            met_y = j1_pt_y + j2_pt_y + j3_pt_y
            met = np.sqrt(met_x * met_x + met_y * met_y)

            dsig = -dsigmoid(-(met-50.0))
            dmet = 0.5 / met

            # wrt J_scale 
            dj1  = -2*met_x*j1_threeM[i, 0] * np.cos(j1_threeM[i, 2]) / J_scale**2
            dj1 += -2*met_y*j1_threeM[i, 0] * np.sin(j1_threeM[i, 2]) / J_scale**2
            # wrt j_scale
            dj2j3  = -2*met_x*j2_threeM[i, 0] * np.cos(j2_threeM[i, 2]) / j_scale**2
            dj2j3 += -2*met_x*j3_threeM[i, 0] * np.cos(j3_threeM[i, 2]) / j_scale**2
            dj2j3 += -2*met_y*j2_threeM[i, 0] * np.sin(j2_threeM[i, 2]) / j_scale**2
            dj2j3 += -2*met_y*j3_threeM[i, 0] * np.sin(j3_threeM[i, 2]) / j_scale**2

            dmet_dj1   += dsig * dmet * dj1
            dmet_dj2j3 += dsig * dmet * dj2j3

    return [dmet_dj1 / count_pTcut, dmet_dj2j3 / count_pTcut]


In [9]:
dX_train = X_train
dy_train_sig = np.array([dEff_MET50_sigmoid(dX_train[i][0], 
                                            dX_train[i][1]) for i in range(len(dX_train))])

In [10]:
def minimizer(fun, bounds):
    return hybrid_minimizer(fun, bounds, 
                            N_rand=2, 
                            N_brute=1)

In [11]:
bayes_der_hyb = UncertaintyOptimization(fun=Eff_MET50_sigmoid,
                                dfun=dEff_MET50_sigmoid,
                                ignore_convergence_warnings=True,
                                verbose=True,
                                param_bounds={'J_scale': (X_lower, X_upper), 
                                              'j_scale': (X_lower, X_upper)},
                                random_state=123)
bayes_der_hyb.minimize_uncertainty(params_train=X_train,
                               minimizer=minimizer,
                           params_val=X_test,
                           niters=8,
                           minimizer_restarts=5,
                           gp_optimizer_restarts=10)

plot_gpr_evolution(bayes=copy.deepcopy(bayes_der_hyb), 
                   X_lower=X_lower, X_upper=X_upper,
                   saveto="derivative_evol")
plot_gpr_iterloss(bayes=copy.deepcopy(bayes_der_hyb),
                  saveto="derivative_loss")

| Iter | J_scale   | j_scale   | Target    | MSE val   | Uncert val|
| 1    | 0.900000  | 1.000000  | 0.580745  | 0.030892  | 2.659504  |
| 2    | 1.000000  | 0.900000  | 0.787183  | 0.030892  | 2.659504  |
| 3    | 1.000000  | 1.000000  | 0.780826  | 0.030892  | 2.659504  |
| 4    | 1.000000  | 1.100000  | 0.661358  | 0.030892  | 2.659504  |
| 5    | 1.100000  | 1.000000  | 0.834261  | 0.030892  | 2.659504  |
[0.74918696 1.29589485]
[(0.7, 0.8491869630674865), (1.1958948509895824, 1.3)]
[1.22020178 0.88116365]
[(1.1202017834295825, 1.3), (0.7811636472371711, 0.9811636472371711)]
[0.80555017 1.25828531]
[(0.7055501730133188, 0.9055501730133187), (1.158285306816844, 1.3)]
[1.08877405 0.77844569]
[(0.988774049041173, 1.1887740490411731), (0.7, 0.8784456870295877)]
[0.76734226 1.12804527]
[(0.7, 0.8673422588620051), (1.0280452711777661, 1.2280452711777663)]
| 6    | 0.700000  | 1.300000  | 0.001049  | 0.009213  | 1.920522  |
[0.93423298 0.77670768]
[(0.8342329754744258, 1.0342329754744257

KeyboardInterrupt: 